In [ ]:
# Data file name defining the batch of reports
data_file = "2026-heatwave-reports.csv"

# Export format for the chart:
# PNG     - export as PNG image
# PDF     - export as PDF file
# <blank> - do not export
export_format = "png"

In [ ]:
%run pathutils.ipynb
%run database.ipynb
%run export.ipynb

In [ ]:
def load_data(year, location, category, species):
    # Construct the query
    query = construct_query("year_to_year_comparison.sql", {
        "YEAR": year,
        "LOCATION": location.replace("'", "''"),
        "CATEGORY": category.replace("'", "''"),
        "SPECIES": species.replace("'", "''")
    })

    # Connect to the database and execute the query and read the results into a dataframe
    df = query_data(query)

    # Check there is some data
    if not df.shape[0]:
        message = f"No data found for species '{species}', category '{category}' at location '{location}' durng '{year}'"
        raise ValueError(message)

    # Index sightings by month and label the series with its year
    df = df.assign(Month=df["Month"].astype(str).str.zfill(2)).set_index("Month")["Sightings"].rename(str(year))
    return df

In [ ]:
import matplotlib.pyplot as plt

def export_comparison_chart(df, location, category, species, export_folder_path, export_file_name, export_format):
    ax = df.plot.bar(
        figsize=(8, 4),
        width=0.8,
        rot=0
    )

    ax.set_xlabel("Month")
    ax.set_ylabel("Sightings")

    title = f"Year-to-Year Comparison : {location}"
    if species:
        title += f", {category}"
    if species:
        title += f", {species}"
    ax.set_title(title)

    # Place the legend underneath the chart
    ax.legend(
        title="Year",
        loc="upper center",
        bbox_to_anchor=(0.5, -0.15),
        ncol=2,
        frameon=False,
    )

    plt.tight_layout()

    # Export the chart
    export_chart(export_folder_path, export_file_name, export_format)

    plt.close()

In [ ]:
from pathlib import Path
import pandas as pd

# Load the report batch definitions : This must be in the reports "data" folder
definitions_csv = Path(get_reports_root_folder()) / "data" / data_file
reports_df = pd.read_csv(definitions_csv)
display(reports_df)
reports_df = reports_df.astype(object).fillna("")
display(reports_df)

In [ ]:
# Create the folder to hold exported reports
export_folder_path = get_export_folder_path()

# Iterate over the reports
for comparison_year, year_of_interest, location, category, species in reports_df.itertuples(index=False, name=None):
    print(f"{year_of_interest} to {comparison_year} comparison: {location}, {category}, {species}")

    # Load the data for the two years and combine them into one dataset
    comparison_df = load_data(comparison_year, location, category, species)
    year_of_interest_df = load_data(year_of_interest, location, category, species)
    combined_df = pd.concat([comparison_df, year_of_interest_df],axis=1).reindex([f"{month:02d}" for month in range(1, 13)])
    display(combined_df)
    # Construct the export file name
    clean_location = clean_string(location)
    clean_category = clean_string(category)
    clean_species = clean_string(species)

    export_file_name = f"{comparison_year}-{year_of_interest}-{clean_location}"
    if clean_category:
        export_file_name += f"-{clean_category}"
    if clean_species:
        export_file_name += f"-{clean_species}"

    # Export the combined data to a spreadsheet
    export_to_spreadsheet(export_folder_path, f"{export_file_name}.xlsx", {
        "Comparison": combined_df
    })

    # Plot and export the chart
    export_comparison_chart(combined_df, location, category, species, export_folder_path, export_file_name, export_format)